In [5]:
from langchain.chat_models import ChatOpenAI
from langchain.agents import initialize_agent, AgentType
from langchain.tools import StructuredTool

llm = ChatOpenAI(temperature=1)

prompt = "Cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $552.00 + $76.16 + $29.12"
# llm.invoke(prompt) ## 제대로된 값을 얻을 수 없음 

def plus(a,b):
    return a + b

## tool은 여러개가 들어갈 수 있다.
agent = initialize_agent(
    llm=llm, 
    verbose=True,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION, 
    tools=[
    StructuredTool.from_function(
        func=plus,
        name="Sum Calculator",
        description="Use this to perform sums of two numbers. This tool take two arguments, both should be numbers."
    )
])

agent.invoke(prompt)




> Entering new AgentExecutor chain...
Thought: I can use the Sum Calculator tool to calculate the total cost of all these expenses by adding them together.
Action:
```
{
  "action": "Sum Calculator",
  "action_input": {
    "a": 355.39,
    "b": 924.87
  }
}
```

Observation: 1280.26
Thought:Now, let's continue adding the remaining expenses to the total.
Action:
```
{
  "action": "Sum Calculator",
  "action_input": {
    "a": 1280.26,
    "b": 721.2
  }
}
```
Observation: 2001.46
Thought:Now, let's add the next expense to the total.
Action:
```
{
  "action": "Sum Calculator",
  "action_input": {
    "a": 2001.46,
    "b": 1940.29
  }
}
```
Observation: 3941.75
Thought:Now, let's add the next expense to the total.
Action:
```
{
  "action": "Sum Calculator",
  "action_input": {
    "a": 3941.75,
    "b": 573.63
  }
}
```
Observation: 4515.38
Thought:Now, let's add the next expense to the total.
Action:
```
{
  "action": "Sum Calculator",
  "action_input": {
    "a": 4515.38,
    "b": 6

{'input': 'Cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $552.00 + $76.16 + $29.12',
 'output': 'The total cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $552.00 + $76.16 + $29.12 is $5273.38.'}

In [ ]:
## ZERO_SHOT_REACT_DESCRIPTION
# 하나의 입력 값을 갖는 함수에만 동작한다
from langchain.tools import Tool

llm = ChatOpenAI(temperature=1)

prompt = "Cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $552.00 + $76.16 + $29.12"

def plus(inputs):
    a, b = inputs.split(",")
    return float(a) + float(b)

agent = initialize_agent(
    llm=llm, 
    verbose=True,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    tools=[
        Tool.from_function(
        func=plus,
        name="Sum Calculator",
        description="Use this to perform sums of two numbers. This tool by sending a pair of numbers seperated by comma. \n Example:1,2"
    )
])

agent.invoke(prompt)




> Entering new AgentExecutor chain...
I need to find the sum of all these numbers.
Action: Sum Calculator
Action Input: 355.39,924.87
Observation: 1280.26
Thought:I need to continue summing up the remaining numbers. 
Action: Sum Calculator
Action Input: 1280.26,721.2
Observation: 2001.46
Thought:I will continue summing up the remaining numbers.
Action: Sum Calculator
Action Input: 2001.46,1940.29
Observation: 3941.75
Thought:I need to keep adding the remaining numbers.
Action: Sum Calculator
Action Input: 3941.75,573.63
Observation: 4515.38
Thought:I will add the rest of the numbers.
Action: Sum Calculator
Action Input: 4515.38,65.72
Observation: 4581.1
Thought:I will continue summing up the remaining numbers. 
Action: Sum Calculator
Action Input: 4581.1,35.00
Observation: 4616.1
Thought:I only have 3 more numbers to add. 
Action: Sum Calculator
Action Input: 4616.1,552.00
Observation: 5168.1
Thought:I have 2 more numbers to sum. 
Action: Sum Calculator
Action Input: 5168.1,76.16
Obs

{'input': 'Cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $552.00 + $76.16 + $29.12',
 'output': '$5273.38'}

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.agents import initialize_agent, AgentType
from langchain.tools import BaseTool
from pydantic import BaseModel, Field
from typing import Type

llm = ChatOpenAI(temperature=1)

prompt = "Cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $552.00 + $76.16 + $29.12"

# def plus(a,b):
#     return a + b

class CalculatorToolArgsSchema(BaseModel):
 a: float = Field(description="The first number")
 b: float = Field(description="The second number")


class CalculatorTool(BaseTool):
    name:Type[str]="CalculatorTool"
    description:Type[str]="""
    Use this to perform sums of two numbers.
    The first and second arguments should be numbers.
    Only receives two arguments.
    """
    args_schema: Type[CalculatorToolArgsSchema] = CalculatorToolArgsSchema
    
    def _run(self, a, b):
       return a + b

agent = initialize_agent(
    llm=llm, 
    verbose=True,
    agent=AgentType.OPENAI_FUNCTIONS,
    tools=[
        CalculatorTool()
    ]
)

agent.invoke(prompt)


## 위의 agent에 비해 prompt의 양이 현저히 줄었다.
## 위에서는 "이전에 계산한 합은 ***야. 여기에 **값을 더해" 와 같이 이전의 계산 합을 프롬프트에서 전해줬어야 했는데
## openAI function agent로 pydantic을 사용하여 arguments schema 를 지정하여 쓸모없는 프롬프트를 줄일 수 있음



> Entering new AgentExecutor chain...

Invoking: `CalculatorTool` with `{'a': 355.39, 'b': 924.87}`


1280.26
Invoking: `CalculatorTool` with `{'a': 1280.26, 'b': 721.2}`


2001.46
Invoking: `CalculatorTool` with `{'a': 2001.46, 'b': 1940.29}`


3941.75
Invoking: `CalculatorTool` with `{'a': 3941.75, 'b': 573.63}`


4515.38
Invoking: `CalculatorTool` with `{'a': 4515.38, 'b': 65.72}`


4581.1
Invoking: `CalculatorTool` with `{'a': 4581.1, 'b': 35.0}`


4616.1
Invoking: `CalculatorTool` with `{'a': 4616.1, 'b': 552.0}`


5168.1
Invoking: `CalculatorTool` with `{'a': 5168.1, 'b': 76.16}`


5244.26
Invoking: `CalculatorTool` with `{'a': 5244.26, 'b': 29.12}`


5273.38The total cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $552.00 + $76.16 + $29.12 is $5273.38.

> Finished chain.


{'input': 'Cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $552.00 + $76.16 + $29.12',
 'output': 'The total cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $552.00 + $76.16 + $29.12 is $5273.38.'}

In [22]:
from langchain.chat_models import ChatOpenAI
from langchain.agents import initialize_agent, AgentType
from langchain.tools import BaseTool
from pydantic import BaseModel, Field
from typing import Type
from langchain.utilities import DuckDuckGoSearchAPIWrapper

llm = ChatOpenAI(temperature=1, model="gpt-4o-mini")

class StockMarketSymbolSearchToolArgsSchema(BaseModel):
    query: str = Field(description="The query you will search for")



class StockMarketSymbolSearchTool(BaseTool):
    name:Type[str]="StockMarketSymbolSearchTool"
    description:Type[str]="""
    Use this to find the stock market symbol for a company.
    It takes a query as an argument.
    Example query: Stock Market Symbol for Apple Company.    
    """
    args_schema: Type[StockMarketSymbolSearchToolArgsSchema] = StockMarketSymbolSearchToolArgsSchema
    
    def _run(self, query):
        ddg = DuckDuckGoSearchAPIWrapper()
        return ddg.run(query)


agent = initialize_agent(
    llm=llm, 
    verbose=True,
    agent=AgentType.OPENAI_FUNCTIONS,
    tools=[
        StockMarketSymbolSearchTool()
    ]
)

prompt = "Give me information on CloudFlare's stock and help me analyze if it's a potential good investment. Also tell me what symbol does the stock have."
agent.invoke(prompt)



> Entering new AgentExecutor chain...

Invoking: `StockMarketSymbolSearchTool` with `{'query': 'CloudFlare'}`


您遇到的问题是由 Cloudflare 提供的安全保护机制引发的。Cloudflare 是一家提供网站性能和安全服务的公司，它为网站提供 DDoS  … Nov 18, 2025 · Cloudflare 人称“赛博活佛”，以其量大管饱的免费计划广受独立开发者喜爱。 现在用 Cloudflare 的人挺多的，特别 … 是在今天的 2024 开发者周上，Cloudflare 发布了一系列令人激动的新特性，例如 Python Worker 以及 Workers AI，把应用开发与交 … 我的域名的DNS服务器已设置，询问注册局已在全球生效，但是，我的cloudflare仍然无法验证我的名称服务器。 如何评价cloudflare于2025年11月18日出现全球性网络故障? 北京时间19:48cloudflare服务状态显示出现全球性网络故障 21:09 …

/Users/dasom/Desktop/langchain-gpt/env/lib/python3.11/site-packages/langchain/utilities/duckduckgo_search.py:43: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



Invoking: `StockMarketSymbolSearchTool` with `{'query': 'Cloudflare'}`
responded: Cloudflare is a well-known company that provides a suite of services for enhancing website performance and security, including DDoS protection and content delivery networks (CDN). 

To find the stock market symbol for Cloudflare, I will try again since the previous result did not provide the necessary information. Let's fetch the stock market symbol for Cloudflare once more.



/Users/dasom/Desktop/langchain-gpt/env/lib/python3.11/site-packages/langchain/utilities/duckduckgo_search.py:43: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Dec 18, 2025 · Cloudflare에서 Saltstack의 기초와 이를 사용하는 방법을 알아봅니다. 또한 Cloudflare가 인프라를 구축하여 에지에서의 Salt 장애로 인한 릴리스 지연을 5% 이상 줄인 방법도 설명합니다. ... Oct 30, 2025 · At Cloudflare, we keep your personal information personal and private. We will not sell, rent, share, or otherwise …It seems I am having trouble retrieving the stock market symbol for Cloudflare. However, I can tell you that Cloudflare trades under the symbol **NET** on the New York Stock Exchange (NYSE).

### Investment Analysis of Cloudflare (NET)

1. **Business Model**: Cloudflare operates a subscription-based model, providing various services that offer increasing value over time. This includes security, performance optimization, and domain services which are essential for businesses running applications online.

2. **Market Position**: Cloudflare has established a strong position in the content delivery and cybersecurity industries, competing with other well-known companies like Akamai, Fastly, and Amazon Web Services.

3. **Financ

{'input': "Give me information on CloudFlare's stock and help me analyze if it's a potential good investment. Also tell me what symbol does the stock have.",
 'output': "It seems I am having trouble retrieving the stock market symbol for Cloudflare. However, I can tell you that Cloudflare trades under the symbol **NET** on the New York Stock Exchange (NYSE).\n\n### Investment Analysis of Cloudflare (NET)\n\n1. **Business Model**: Cloudflare operates a subscription-based model, providing various services that offer increasing value over time. This includes security, performance optimization, and domain services which are essential for businesses running applications online.\n\n2. **Market Position**: Cloudflare has established a strong position in the content delivery and cybersecurity industries, competing with other well-known companies like Akamai, Fastly, and Amazon Web Services.\n\n3. **Financial Performance**: \n   - Historically, Cloudflare has shown strong revenue growth, often 

### Alpha Ventage를 사용하여 데이터 가져오기

In [40]:
import os
import toml
import requests
from langchain.chat_models import ChatOpenAI
from langchain.agents import initialize_agent, AgentType
from langchain.tools import BaseTool
from pydantic import BaseModel, Field
from typing import Type
from langchain.utilities import DuckDuckGoSearchAPIWrapper

llm = ChatOpenAI(temperature=1
                 , model_name="gpt-4-turbo-2024-04-09"
                 )

secrets = toml.load(".streamlit/secrets.toml")
alpha_ventage_api_key = secrets.get("ALPHA_VANTAGE_API_KEY")

class StockMarketSymbolSearchToolArgsSchema(BaseModel):
    query: str = Field(description="The query you will search for. Example query: Stock Market Symbol fro Apple Company")

class StockMarketSymbolSearchTool(BaseTool):
    name:Type[str]="StockMarketSymbolSearchTool"
    description:Type[str]="""
    Use this to find the stock market symbol for a company.
    It takes a query as an argument.
    Example query: Stock Market Symbol for Apple Company.    
    """
    args_schema: Type[StockMarketSymbolSearchToolArgsSchema] = StockMarketSymbolSearchToolArgsSchema
    
    def _run(self, query):
        ddg = DuckDuckGoSearchAPIWrapper()
        return ddg.run(query)
    
class CompanyOverviewToolArgsSchema(BaseModel):
    symbol: str = Field(description="Stock symbol of the company. Example: AAPL, TSLA")

class CompanyOverviewTool(BaseTool):
    name:Type[str]="CompanyOverview"
    description:Type[str]="""
    Use this to get an overview of the financials of the company.
    You should enter a stock symbol.
    """
    args_schema: Type[CompanyOverviewToolArgsSchema] = CompanyOverviewToolArgsSchema
    
    def _run(self, symbol):
        r = requests.get(f"https://www.alphavantage.co/query?function=OVERVIEW&symbol={symbol}&apikey={alpha_ventage_api_key}")
        return r.json()

class CompanyIncomeStatementTool(BaseTool):
    name:Type[str]="CompanyIncomeStatement"
    description:Type[str]="""
    Use this to get the income statement of the company.
    You should enter a stock symbol.
    """
    args_schema: Type[CompanyOverviewToolArgsSchema] = CompanyOverviewToolArgsSchema
    
    def _run(self, symbol):
        r = requests.get(f"https://www.alphavantage.co/query?function=INCOME_STATEMENT&symbol={symbol}&apikey={alpha_ventage_api_key}")
        return r.json()

class CompanyStockPerformanceTool(BaseTool):
    name:Type[str]="CompanyStockPerformance"
    description:Type[str]="""
    Use this to get the weekly performance of a company stock.
    You should enter a stock symbol.
    """
    args_schema: Type[CompanyOverviewToolArgsSchema] = CompanyOverviewToolArgsSchema
    
    def _run(self, symbol):
        r = requests.get(f"https://www.alphavantage.co/query?function=TIME_SERIES_WEEKLY&symbol={symbol}&apikey={alpha_ventage_api_key}")
        return r.json()

agent = initialize_agent(
    llm=llm, 
    verbose=True,
    agent=AgentType.OPENAI_FUNCTIONS,
    tools=[
        StockMarketSymbolSearchTool(),
        CompanyOverviewTool(),
        CompanyIncomeStatementTool(),
        CompanyStockPerformanceTool()
    ]
)

prompt = """
Give me financial information on CloudFlare's stock, considering its financials, income statements and stock performance and help me analyze if it's a potential good investment. 
"""
agent.invoke(prompt)



> Entering new AgentExecutor chain...

Invoking: `StockMarketSymbolSearchTool` with `{'query': 'Stock Market Symbol for CloudFlare'}`


3 days ago · A detailed overview of Cloudflare, Inc. (NET) stock, including real-time price, chart, key statistics, news, and more. 6 days ago · Real-time Cloudflare stock price (NYSE: NET) plus other valuable data points like day range, year, stock analyst insights, related … Nov 18, 2025 · What is the ticker symbol for Cloudflare, Inc. Class A Common Stock? The ticker symbol for Cloudflare, Inc. Class A … 4 days ago · Latest Cloudflare Inc (NET:NYQ) share price with interactive charts, historical prices, comparative analysis, forecasts, … 3 days ago · Complete Cloudflare Inc. Cl A stock information by Barron's. View real-time NET stock price and news, along with industry-best …

/Users/dasom/Desktop/langchain-gpt/env/lib/python3.11/site-packages/langchain/utilities/duckduckgo_search.py:43: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



Invoking: `CompanyOverview` with `{'symbol': 'NET'}`


{'Symbol': 'NET', 'AssetType': 'Common Stock', 'Name': 'Cloudflare Inc', 'Description': "Cloudflare Inc. (ticker: NET) is a premier global cloud services provider headquartered in San Francisco, California, specializing in security, performance, and reliability for websites and applications. With an extensive suite of offerings ranging from content delivery networks (CDN) to DDoS mitigation and advanced internet security solutions, Cloudflare serves millions of clients across diverse sectors. As organizations increasingly prioritize secure and efficient internet infrastructure, Cloudflare's innovative approach and scalable solutions position it as a critical player in the rapidly evolving digital ecosystem.", 'CIK': '1477333', 'Exchange': 'NYSE', 'Currency': 'USD', 'Country': 'USA', 'Sector': 'TECHNOLOGY', 'Industry': 'SOFTWARE - INFRASTRUCTURE', 'Address': '101 TOWNSEND STREET, SAN FRANCISCO, CA, UNITED STATES, 94107', 'OfficialS

Retrying langchain.chat_models.openai.ChatOpenAI.completion_with_retry.<locals>._completion_with_retry in 4.0 seconds as it raised RateLimitError: Rate limit reached for gpt-4-turbo-preview in organization org-eoll84qEHJ88ZFtrLiafSPIH on tokens per min (TPM): Limit 30000, Used 10742, Requested 20057. Please try again in 1.598s. Visit https://platform.openai.com/account/rate-limits to learn more..


### Overview of Cloudflare Inc. (NET)
- **Sector:** Technology
- **Industry:** Software - Infrastructure
- **Exchange:** NYSE
- **Country:** USA
- **Currency:** USD
- **Market Capitalization:** $70.89 Billion
- **Official Site:** [Cloudflare](https://www.cloudflare.com)

### Financial Performance

#### Latest Financials - Company Income Statement
- **Annual Gross Profit (2024):** $1.29 Billion
- **Annual Total Revenue (2024):** $1.67 Billion
- **Annual Operating Expenses (2024):** $1.45 Billion
- **Annual Net Income (2024):** -$78.8 Million

- **Growth in Total Revenue:** $1.67 Billion in 2024, up 22% from $1.30 Billion in 2023
- **Growth Trajectory:** Revenue has consistently increased year over year.

#### Fiscal Health
- **Book Value per Share:** $3.846
- **EPS (Earnings Per Share):** -$0.30
- **EBITDA:** $62.05 Million

- The company is not paying dividends currently, typical for growth-oriented tech companies focusing on expansion.

### Stock Performance
Cloudflare's stock price h

{'input': "\nGive me financial information on CloudFlare's stock, considering its financials, income statements and stock performance and help me analyze if it's a potential good investment. \n",
 'output': "### Overview of Cloudflare Inc. (NET)\n- **Sector:** Technology\n- **Industry:** Software - Infrastructure\n- **Exchange:** NYSE\n- **Country:** USA\n- **Currency:** USD\n- **Market Capitalization:** $70.89 Billion\n- **Official Site:** [Cloudflare](https://www.cloudflare.com)\n\n### Financial Performance\n\n#### Latest Financials - Company Income Statement\n- **Annual Gross Profit (2024):** $1.29 Billion\n- **Annual Total Revenue (2024):** $1.67 Billion\n- **Annual Operating Expenses (2024):** $1.45 Billion\n- **Annual Net Income (2024):** -$78.8 Million\n\n- **Growth in Total Revenue:** $1.67 Billion in 2024, up 22% from $1.30 Billion in 2023\n- **Growth Trajectory:** Revenue has consistently increased year over year.\n\n#### Fiscal Health\n- **Book Value per Share:** $3.846\n- *